# GPT-5.6 Responses API tutorial: deployment health check

This notebook teaches the core Responses API flow with one deterministic deployment-health workload. It focuses on API mechanics, not on benchmarking reasoning quality or cost.

**Audience:** Python developers who know basic API calls and want to learn response items, stateful continuation, function calling, and tool-result handling.

**Prerequisites**

- Python 3.12
- Run `uv sync` from the project root, then start Jupyter with `uv run jupyter lab`
- Access to `gpt-5.6`
- A project-root `.env.local` file containing `OPENAI_API_KEY`

**Learning goals**

1. Send a reasoning request and read `response.output_text`.
2. Inspect typed response items without assuming a fixed output index.
3. Continue a stored response with `previous_response_id`.
4. Complete a function-call round trip with the matching `call_id`.
5. Validate the response chain and tool observation with protocol assertions.

Official references: [GPT-5.6 model guidance](https://developers.openai.com/api/docs/guides/latest-model), [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state), and [Function calling](https://developers.openai.com/api/docs/guides/function-calling).

## Workload: Deployment health check

Imagine that a release engineer has just deployed a new version of a service. Before allowing the rollout to continue, the engineer must check whether the deployment is healthy or should be rolled back.

In this tutorial, GPT-5.6 acts as the deployment reviewer:

- **Person's goal:** decide whether a deployment can continue safely.
- **Information needed:** health-check failures, error rate compared with its baseline, and p95 latency compared with its baseline.
- **Tool:** `lookup_deployment` retrieves observed metrics for a deployment ID from a local fixture.
- **Possible decisions:** `rollback` for an unhealthy deployment or `continue` for a healthy deployment.
- **Tutorial cases:** `DEP-204` is intentionally unhealthy; `DEP-205` is the healthy exercise case.

The records are synthetic and deterministic. No real deployment or monitoring system is accessed. The purpose is to make the API flow immediately understandable: the user asks for a review, the model requests the missing metrics through a tool, the application returns them, and the model applies one visible policy to make a decision.

```text
Release engineer → GPT-5.6 → lookup_deployment → observed metrics → rollback / continue
```

## Tutorial outline

1. Set up the client and deterministic fixtures.
2. Ask for general deployment-review guidance.
3. Inspect the response and continue from its ID.
4. Let the model request deployment data through a function.
5. Return the local tool result and receive the final recommendation.
6. Validate the protocol and run a traced exercise on a healthy deployment.

## 1. Setup

`load_api_key()` searches the current directory and its parents for `.env.local`. The client is created directly in the notebook, and the secret is never printed.

The tutorial explicitly requests `all_turns` and checks the effective value returned by GPT-5.6. `low` reasoning effort keeps this small teaching example inexpensive.

In [ ]:
import json

from agents.tracing import flush_traces, response_span, trace
from openai import OpenAI

try:
    from responses_lab import configure_tracing, load_api_key
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Project package not found. Run `uv sync` from the project root and "
        "select its .venv kernel."
    ) from exc

MODEL = "gpt-5.6"
REASONING = {"effort": "low", "context": "all_turns"}

client = OpenAI(api_key=load_api_key())
tracing_setup = configure_tracing()

print(f"Ready to use {MODEL}; the API key was loaded without displaying it.")
print(f"Local trace path: {tracing_setup.local_path}")

## 2. Define the deployment workload

The model can call `lookup_deployment`, but the tool itself is a local Python dictionary. This keeps the notebook deterministic and avoids dependencies on a live monitoring system.

The policy is generic rather than answer-specific: rollback if a health check fails, the error rate exceeds twice its baseline, or p95 latency exceeds 1.5 times its baseline. Otherwise continue. Tool records contain observations, not prewritten conclusions.

In [ ]:
DEPLOYMENTS = {
    "DEP-204": {
        "deployment_id": "DEP-204",
        "service": "checkout-api",
        "version": "2026.08.26.1",
        "error_rate_percent": 4.7,
        "baseline_error_rate_percent": 0.4,
        "p95_latency_ms": 810,
        "baseline_p95_latency_ms": 430,
        "failed_health_checks": 2,
        "total_health_checks": 8,
        "rollback_available": True,
    },
    "DEP-205": {
        "deployment_id": "DEP-205",
        "service": "catalog-api",
        "version": "2026.08.26.2",
        "error_rate_percent": 0.3,
        "baseline_error_rate_percent": 0.4,
        "p95_latency_ms": 295,
        "baseline_p95_latency_ms": 310,
        "failed_health_checks": 0,
        "total_health_checks": 8,
        "rollback_available": True,
    },
}

DEPLOYMENT_POLICY = (
    "You review software deployments. Recommend rollback when any health check fails, "
    "the observed error rate is more than twice its baseline, or observed p95 latency "
    "is more than 1.5 times its baseline. Otherwise recommend continue. Never invent "
    "deployment data. After data is available, cite the observed error rate and p95 "
    "latency and end with exactly `Decision: rollback` or `Decision: continue`."
)

LOOKUP_DEPLOYMENT_TOOL = {
    "type": "function",
    "name": "lookup_deployment",
    "description": "Retrieve observed health metrics for one deployment ID.",
    "parameters": {
        "type": "object",
        "properties": {
            "deployment_id": {
                "type": "string",
                "enum": list(DEPLOYMENTS),
            }
        },
        "required": ["deployment_id"],
        "additionalProperties": False,
    },
    "strict": True,
}
FORCE_LOOKUP = {"type": "function", "name": "lookup_deployment"}

print(json.dumps(DEPLOYMENTS, indent=2))

## 3. Send a basic reasoning request

Start with a general question. We set `store=True` because the next step will continue this response by ID. Use `response.output_text` for assistant text because `response.output` may also contain reasoning or tool items.

In [ ]:
basic_response = client.responses.create(
    model=MODEL,
    instructions=DEPLOYMENT_POLICY,
    input=(
        "Name the three deployment-health signals in our policy and briefly explain "
        "why each one matters. Do not evaluate a specific deployment yet."
    ),
    reasoning=REASONING,
    text={"verbosity": "low"},
    max_output_tokens=300,
    store=True,
)

print(basic_response.output_text)

## 4. Inspect typed response items

Output order is not a stable contract. Inspect `item.type` and search for the item you need instead of assuming that `output[0]` is an assistant message. The helper below keeps output concise and does not print opaque encrypted content.

In [ ]:
print("Response ID:", basic_response.id)
print("Item types:", [item.type for item in basic_response.output])
print("Effective reasoning context:", basic_response.reasoning.context)
print("Usage:", basic_response.usage.model_dump() if basic_response.usage else None)

for item in basic_response.output:
    payload = item.model_dump(exclude={"encrypted_content"})
    print(json.dumps(payload, indent=2)[:1_500])

## 5. Continue with `previous_response_id` and request a tool call

The new user request refers to the same deployment-review goal. Passing the prior response ID creates a stateful chain, while `all_turns` makes the reasoning-continuity intent explicit. The tool choice is forced so the tutorial is deterministic.

In [ ]:
tool_response = client.responses.create(
    model=MODEL,
    instructions=DEPLOYMENT_POLICY,
    previous_response_id=basic_response.id,
    input="Inspect DEP-204 and decide whether to continue or roll back the deployment.",
    tools=[LOOKUP_DEPLOYMENT_TOOL],
    tool_choice=FORCE_LOOKUP,
    reasoning=REASONING,
    max_output_tokens=300,
    store=True,
)

tool_call = next(
    item
    for item in tool_response.output
    if item.type == "function_call" and item.name == "lookup_deployment"
)
arguments = json.loads(tool_call.arguments)

print("Previous response ID:", tool_response.previous_response_id)
print("Tool response ID:", tool_response.id)
print("Output item types:", [item.type for item in tool_response.output])
print("Function:", tool_call.name)
print("Arguments:", arguments)
print("Call ID:", tool_call.call_id)

## 6. Execute the local tool

The API requests a function, but your application executes it. Here the implementation is a dictionary lookup. Keep the returned data observational: the policy and model make the decision.

In [ ]:
deployment_id = arguments["deployment_id"]
deployment_record = DEPLOYMENTS[deployment_id]
tool_output = json.dumps(deployment_record)

print(json.dumps(deployment_record, indent=2))

## 7. Return the function output

Send a `function_call_output` item with the exact `call_id` from the model. Continue from `tool_response.id`, which is the immediately preceding response in this branch.

### Why are both `previous_response_id` and tool output required?

They carry different information:

| Field | Role |
| --- | --- |
| `previous_response_id` | Tells the server **where to continue**. The referenced response contains the model's reasoning and `function_call` request. |
| `function_call_output` | Tells the model **what the application learned** after executing the custom function. This result did not exist when the previous response was created. |
| `call_id` | Tells the server **which pending function call** the new result answers. |

The previous response contains a request like `lookup_deployment(DEP-204)`, not the deployment metrics themselves. The OpenAI server does not execute this notebook's local Python dictionary lookup. After the response is returned, the application executes the function and obtains new observations such as the 4.7% error rate and 810 ms p95 latency. Those observations must be sent back as `function_call_output`.

```text
tool_response.id
└── function_call: lookup_deployment(DEP-204)
    └── call_id: call_...
             ↑
             │ matching call_id
             │
function_call_output
└── observed deployment metrics
```

A useful shorthand is:

```text
previous_response_id = where to attach the new information
function_call_output = the new information to attach
call_id              = the specific tool request being answered
```

If the request supplied only `previous_response_id`, the model would remember that it requested `lookup_deployment`, but it would still not know the function result. It could call the tool again or remain unable to make the deployment decision.

Without `previous_response_id`, a stateless client would have to resend the earlier response output—including the reasoning and function-call items—together with the new `function_call_output`. Stateful continuation avoids resending those earlier items; it does **not** eliminate the need to provide a newly executed custom-tool result. Hosted tools that OpenAI executes are different from this application-defined function.

In [ ]:
final_response = client.responses.create(
    model=MODEL,
    instructions=DEPLOYMENT_POLICY,
    previous_response_id=tool_response.id,
    input=[
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": tool_output,
        }
    ],
    tools=[LOOKUP_DEPLOYMENT_TOOL],
    tool_choice="none",
    reasoning=REASONING,
    text={"verbosity": "low"},
    max_output_tokens=400,
    store=True,
)

print(final_response.output_text)

## 8. Validate the protocol

These assertions verify the API path and the use of tool observations. They are not a model-quality benchmark. Exact IDs, item linkage, and the requested deployment are strict; the final text is checked only for the required decision line and key observed value.

In [ ]:
assert basic_response.id
assert basic_response.output_text
assert basic_response.reasoning.context == "all_turns"
assert tool_response.previous_response_id == basic_response.id
assert tool_call.name == "lookup_deployment"
assert arguments["deployment_id"] == "DEP-204"
assert final_response.previous_response_id == tool_response.id
assert "decision: rollback" in final_response.output_text.lower()
assert "4.7" in final_response.output_text

print("All DEP-204 protocol checks passed.")

## 9. Summarize the three responses

Usage is displayed for observability only. This tutorial does not compare cost or make a savings claim. `previous_response_id` simplifies state management, but previous context still contributes to input usage.

In [ ]:
def usage_summary(label: str, response) -> dict:
    usage = response.usage
    input_details = usage.input_tokens_details if usage else None
    output_details = usage.output_tokens_details if usage else None
    return {
        "step": label,
        "response_id": response.id,
        "previous_response_id": response.previous_response_id,
        "input_tokens": usage.input_tokens if usage else 0,
        "cached_tokens": getattr(input_details, "cached_tokens", 0),
        "cache_write_tokens": getattr(input_details, "cache_write_tokens", 0),
        "output_tokens": usage.output_tokens if usage else 0,
        "reasoning_tokens": getattr(output_details, "reasoning_tokens", 0),
    }

usage_rows = [
    usage_summary("basic", basic_response),
    usage_summary("tool_request", tool_response),
    usage_summary("final", final_response),
]
print(json.dumps(usage_rows, indent=2))

## Exercise: check a healthy deployment

Predict the result for `DEP-205`, then reuse the same two-call tool loop. The expected decision is the opposite of the main example, which helps detect hard-coded recommendations.

The solution function accepts a response-creation function. This lets the answer scaffold run the same API flow inside Trace without adding tracing details to the main tutorial.

In [ ]:
def run_deployment_check(deployment_id: str, create_response) -> dict:
    requested = create_response(
        model=MODEL,
        instructions=DEPLOYMENT_POLICY,
        previous_response_id=basic_response.id,
        input=f"Inspect {deployment_id} and decide whether to continue or roll back.",
        tools=[LOOKUP_DEPLOYMENT_TOOL],
        tool_choice=FORCE_LOOKUP,
        reasoning=REASONING,
        max_output_tokens=300,
        store=True,
    )
    call = next(
        item
        for item in requested.output
        if item.type == "function_call" and item.name == "lookup_deployment"
    )
    called_id = json.loads(call.arguments)["deployment_id"]
    observed = DEPLOYMENTS[called_id]

    completed = create_response(
        model=MODEL,
        instructions=DEPLOYMENT_POLICY,
        previous_response_id=requested.id,
        input=[
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(observed),
            }
        ],
        tools=[LOOKUP_DEPLOYMENT_TOOL],
        tool_choice="none",
        reasoning=REASONING,
        text={"verbosity": "low"},
        max_output_tokens=400,
        store=True,
    )
    return {
        "requested": requested,
        "completed": completed,
        "call": call,
        "called_id": called_id,
        "observed": observed,
    }

### Answer scaffold with Trace

`response_span()` records the actual Responses calls inside one workflow trace. Local JSONL tracing remains available even when dashboard mirroring is disabled.

In [ ]:
def create_traced_response(**request):
    with response_span() as span:
        response = client.responses.create(**request)
        span.span_data.response = response
        span.span_data.usage = response.usage.model_dump() if response.usage else None
        return response

with trace(
    "GPT-5.6 deployment tutorial exercise",
    metadata={"model": MODEL, "deployment_id": "DEP-205"},
) as workflow_trace:
    exercise_result = run_deployment_check("DEP-205", create_traced_response)

flush_traces()
exercise_text = exercise_result["completed"].output_text

assert exercise_result["called_id"] == "DEP-205"
assert exercise_result["completed"].previous_response_id == exercise_result["requested"].id
assert "decision: continue" in exercise_text.lower()

print(exercise_text)
print("Trace ID:", workflow_trace.trace_id)
print("Local trace:", tracing_setup.local_path)
if tracing_setup.openai_dashboard_enabled:
    print("Dashboard: https://platform.openai.com/traces")
else:
    print("Dashboard mirroring is disabled; the local trace remains available.")

## Common pitfalls and extensions

- **Do not assume `response.output[0]` is text.** Search typed items and use `response.output_text` for assistant text.
- **Return the exact `call_id`.** A function result must link to the function call that requested it.
- **Continue from the immediately preceding response.** The function result follows `tool_response.id`, not `basic_response.id`.
- **Repeat important instructions.** Keep the deployment policy explicit on each request in the chain.
- **Do not treat `previous_response_id` as free history.** It simplifies state management, but prior context still contributes to input usage.

Optional extensions for later notebooks include stateless encrypted-item replay, prompt-cache analysis, retained-reasoning comparisons, and Compaction. Those topics are intentionally outside this Tutorial's core scope.

## References: Responses API retained reasoning

This tutorial establishes the stateful function-calling foundation. The official resources below explain how the Responses API can retain reasoning across turns, how to continue a response chain, and how to manage longer-running context.

### Core API documentation

- [GPT-5.6 model guidance](https://developers.openai.com/api/docs/guides/latest-model) — explains `reasoning.context`, including `all_turns` and `current_turn`, and recommends pairing retained reasoning with `previous_response_id` for stateful continuation.
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state) — explains multi-turn response chaining with `previous_response_id` and the relationship between stored state, request history, and token usage.
- [Function calling](https://developers.openai.com/api/docs/guides/function-calling) — shows the tool-call loop, including returning a `function_call_output` associated with the original `call_id`.
- [Compaction](https://developers.openai.com/api/docs/guides/compaction) — explains how to reduce the context carried by long-running conversations while preserving the state needed for later turns.

### Retained-reasoning deep dives and case studies

- [The builder's guide to GPT-5.6](https://openai.com/index/builders-guide-to-gpt-5-6/) — connects retained reasoning and Compaction to long-running agent design and reports a workload-specific efficiency experiment.
- [How two settings tripled our ARC-AGI-3 scores](https://openai.com/index/how-two-settings-tripled-our-arc-agi-3-scores/) — presents a workload-specific case study of retained reasoning and Compaction in a long-horizon task.
- [New tools and features in the Responses API](https://openai.com/index/new-tools-and-features-in-the-responses-api/) — introduces encrypted reasoning items and preserving reasoning across function calls, including stateless or Zero Data Retention workflows.

> These case-study results are evidence for their measured workloads, not a universal cost or quality guarantee. Notebooks 02 and 03 build controlled experiments on top of this tutorial to measure the effect for this repository's own workloads.

## Conclusion

You created a GPT-5.6 Responses API chain, inspected typed output items, continued with `previous_response_id`, executed a local function, returned its result with `call_id`, validated both rollback and continue paths, and captured the exercise with Trace.

Notebook 02 can now build on this API foundation to compare `current_turn` and `all_turns` under a controlled multi-turn workload.